# Grid Demand Forecasting: Training Pipeline
Architecture:
Prophet (Baseline) + LightGBM (Residuals) + Isolation Forest (Anomaly Detection)

### 1. Kualitas Data Engineering
We merge high-frequency daily data with low-frequency macroeconomic monthly data (Forward Filling). 
We also engineer temporal features (`Lag_1`, `Rolling_7`) to give the model short-term memory, and isolate anomalies to prevent overfitting.

In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from prophet import Prophet
from sklearn.ensemble import IsolationForest
from sklearn.metrics import mean_absolute_error, mean_squared_error
import joblib
import warnings
warnings.filterwarnings('ignore')

# 1. Load Data
daily_input_path = 'Outputs/dataset_daily_processed.csv'
monthly_input_path = 'Outputs/dataset_monthly_processed.csv'
df_daily = pd.read_csv(daily_input_path)
df_daily['Date'] = pd.to_datetime(df_daily['Date'])
df_monthly = pd.read_csv(monthly_input_path)

# 2. Merge Data (Forward Fill Macro to Daily)
df_daily['Year'] = df_daily['Date'].dt.year
df_daily['Month'] = df_daily['Date'].dt.month
macro_columns = ['Year', 'Month', 'GDP', 'Population', 'Industrial_Index']
df_macro = df_monthly[[col for col in macro_columns if col in df_monthly.columns]]
df = df_daily.merge(df_macro, on=['Year', 'Month'], how='left').drop(['Year', 'Month'], axis=1)

# 3. Define Features
exogenous_features = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday', 
    'Avg_Temp', 'Rainfall', 'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7',
    'GDP', 'Population', 'Industrial_Index'
]
target_col = 'Demand_MWh'
df = df.dropna(subset=exogenous_features + [target_col]).copy()

### 2. Metodologi & Logika (Train/Test Split & Training)
We split the data chronologically (80% Train, 20% Test) to validate the model's ability to generalize to future, unseen data without data leakage.

In [ ]:
# Chronological Train/Test Split
train_size = int(len(df) * 0.8)
train_df, test_df = df.iloc[:train_size].copy(), df.iloc[train_size:].copy()

# A. Train Prophet (Baseline)
df_prophet_train = train_df[['Date', target_col]].rename(columns={'Date': 'ds', target_col: 'y'})
prophet_model = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False)
prophet_model.fit(df_prophet_train)

# Calculate Residuals for LightGBM
train_df['Prophet_Pred'] = prophet_model.predict(df_prophet_train[['ds']])['yhat'].values
train_df['Residuals'] = train_df[target_col] - train_df['Prophet_Pred']

# B. Train LightGBM (Residuals)
# Hyperparameters chosen for a balance of learning speed and preventing overfitting (max_depth=7)
lgbm_model = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=7, random_state=42)
lgbm_model.fit(train_df[exogenous_features], train_df['Residuals'])

# C. Train Isolation Forest (Anomalies)
iso_forest = IsolationForest(n_estimators=100, contamination=0.02, random_state=42)
iso_forest.fit(train_df[exogenous_features])

### 3. Metrik Performa (Model Evaluation)
Evaluating the Hybrid Model on the 20% unseen test data.

In [ ]:
# 1. Prophet Prediction on Test
df_prophet_test = test_df[['Date']].rename(columns={'Date': 'ds'})
test_df['Prophet_Pred'] = prophet_model.predict(df_prophet_test)['yhat'].values

# 2. LightGBM Prediction on Test
test_df['LGBM_Residual_Pred'] = lgbm_model.predict(test_df[exogenous_features])

# 3. Final Hybrid Prediction
test_df['Hybrid_Prediction'] = test_df['Prophet_Pred'] + test_df['LGBM_Residual_Pred']

# Calculate Metrics
mae = mean_absolute_error(test_df[target_col], test_df['Hybrid_Prediction'])
rmse = np.sqrt(mean_squared_error(test_df[target_col], test_df['Hybrid_Prediction']))

# MAPE requires avoiding division by zero
mape = np.mean(np.abs((test_df[target_col] - test_df['Hybrid_Prediction']) / test_df[target_col])) * 100

print("--- HYBRID MODEL PERFORMANCE (UNSEEN TEST DATA) ---")
print(f"MAE  (Mean Absolute Error): {mae:.2f} MWh")
print(f"RMSE (Root Mean Sq Error) : {rmse:.2f} MWh")
print(f"MAPE (Mean Abs % Error)   : {mape:.2f}%")

# Save Models for Inference
os.makedirs('Models', exist_ok=True)
joblib.dump(prophet_model, 'Models/prophet_model.joblib')
joblib.dump(lgbm_model, 'Models/lgbm_model.joblib')
joblib.dump(iso_forest, 'Models/iso_forest.joblib')
print("\n Models successfully saved to /Models directory.")